In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

In [2]:
def diebold_mariano_test(actual, pred1, pred2):
    """
    Computes the DM test statistic for absolute errors.
    pred1 = Proposed Model (TFT)
    pred2 = Rival Model
    """
    e1 = np.abs(actual - pred1)
    e2 = np.abs(actual - pred2)
    d = e1 - e2
    
    mean_d = np.mean(d)
    var_d = np.var(d, ddof=1)
    
    # Handle edge case where predictions are identical
    if var_d == 0:
        return 0.0, 1.0 
    
    dm_stat = mean_d / np.sqrt(var_d / len(d))
    p_value = 2 * (1 - stats.norm.cdf(np.abs(dm_stat)))
    return dm_stat, p_value

In [3]:
print("Loading data files...")
tft_df = pd.read_csv('tft_item_level (2).csv')
xgb_df = pd.read_csv('xgb_aggr.csv')
prophet_df = pd.read_csv('prophet_agg.csv')
seq2seq_df = pd.read_csv('seq2seq_item_level.csv')

Loading data files...


In [4]:
# Get the universal list of 50 items
items = tft_df['item_id'].unique()
items

array(['FOODS_1_004', 'FOODS_1_012', 'FOODS_1_018', 'FOODS_1_019',
       'FOODS_1_021', 'FOODS_1_032', 'FOODS_1_036', 'FOODS_1_040',
       'FOODS_1_042', 'FOODS_1_043', 'FOODS_1_044', 'FOODS_1_045',
       'FOODS_1_046', 'FOODS_1_054', 'FOODS_1_055', 'FOODS_1_058',
       'FOODS_1_067', 'FOODS_1_082', 'FOODS_1_085', 'FOODS_1_086',
       'FOODS_1_087', 'FOODS_1_099', 'FOODS_1_104', 'FOODS_1_110',
       'FOODS_1_112', 'FOODS_1_113', 'FOODS_1_120', 'FOODS_1_130',
       'FOODS_1_137', 'FOODS_1_139', 'FOODS_1_153', 'FOODS_1_154',
       'FOODS_1_161', 'FOODS_1_162', 'FOODS_1_163', 'FOODS_1_166',
       'FOODS_1_170', 'FOODS_1_172', 'FOODS_1_173', 'FOODS_1_181',
       'FOODS_1_183', 'FOODS_1_184', 'FOODS_1_194', 'FOODS_1_200',
       'FOODS_1_201', 'FOODS_1_204', 'FOODS_1_206', 'FOODS_1_217',
       'FOODS_1_218', 'FOODS_1_219'], dtype=object)

In [5]:
# Define the rivals and their respective prediction columns
rival_models = {
    'XGBoost': (xgb_df, 'xgb_pred'),
    'Prophet': (prophet_df, 'prophet_pred'),
    'Seq2Seq': (seq2seq_df, 'seq2seq_pred')
}

In [6]:
# Executing the test on 150 items(50 x 3 rivals)
results = []
alpha = 0.05 # 95% Confidence Level

print("Running 150 Diebold-Mariano tests...")
for model_name, (rival_df, pred_col) in rival_models.items():
    tft_wins = 0
    rival_wins = 0
    ties = 0
    
    for item in items:
        # Extract the 28-day arrays for the specific item
        actual = tft_df[tft_df['item_id'] == item]['actual'].values
        tft_p = tft_df[tft_df['item_id'] == item]['tft_pred'].values
        rival_p = rival_df[rival_df['item_id'] == item][pred_col].values
        
        # Run DM Test
        dm_stat, p_val = diebold_mariano_test(actual, tft_p, rival_p)
        
        # Tally the statistical winners
        if p_val < alpha:
            if dm_stat < 0:
                tft_wins += 1    # TFT error was significantly smaller
            else:
                rival_wins += 1  # Rival error was significantly smaller
        else:
            ties += 1            # No statistically significant difference
            
    # Record the final tallies for this rival
    results.append({
        'Rival Model': model_name,
        'TFT Better': tft_wins,
        'Rival Better': rival_wins, 
        'Tie (No Sig. Diff)': ties
    })

Running 150 Diebold-Mariano tests...


In [7]:
dm_report = pd.DataFrame(results)

print("\n" + "="*70)
print(" FINAL DIEBOLD-MARIANO TEST RESULTS (ITEM-LEVEL) ")
print("="*70)
print("Hypothesis: Alpha = 0.05 (95% Confidence)")
print("Tested across 50 distinct items (28-day forecast horizon)\n")
print(dm_report.to_string(index=False))
print("="*70)


 FINAL DIEBOLD-MARIANO TEST RESULTS (ITEM-LEVEL) 
Hypothesis: Alpha = 0.05 (95% Confidence)
Tested across 50 distinct items (28-day forecast horizon)

Rival Model  TFT Better  Rival Better  Tie (No Sig. Diff)
    XGBoost           2            17                  31
    Prophet           3            15                  32
    Seq2Seq           4            27                  19


In [10]:
dm_report.to_csv('dm_test_results.csv', index=False)

In [9]:
import pandas as pd

tft_df     = pd.read_csv('tft_item_level (2).csv')
xgb_df     = pd.read_csv('xgb_aggr.csv')
prophet_df = pd.read_csv('prophet_agg.csv')      # new item-level file
seq2seq_df = pd.read_csv('seq2seq_item_level.csv')

print("TFT columns     :", tft_df.columns.tolist())
print("XGBoost columns :", xgb_df.columns.tolist())
print("Prophet columns :", prophet_df.columns.tolist())
print("Seq2Seq columns :", seq2seq_df.columns.tolist())

print("\nTFT shape     :", tft_df.shape)
print("XGBoost shape :", xgb_df.shape)
print("Prophet shape :", prophet_df.shape)
print("Seq2Seq shape :", seq2seq_df.shape)

TFT columns     : ['date', 'item_id', 'actual', 'tft_pred']
XGBoost columns : ['date', 'item_id', 'actual', 'xgb_pred']
Prophet columns : ['date', 'item_id', 'actual', 'prophet_pred']
Seq2Seq columns : ['Unnamed: 0', 'date', 'item_id', 'actual', 'seq2seq_pred']

TFT shape     : (1400, 4)
XGBoost shape : (1400, 4)
Prophet shape : (1400, 4)
Seq2Seq shape : (1400, 5)
